# Deep Learning IDS Using LSTM — Kaggle

**Project:** Design and Implementation of a Deep Learning Intrusion Detection System Using LSTM
**Author:** Kayode Timileyin Nicholas | FUTA Cybersecurity Final Year Project

---

### Setup Instructions (one-time)
1. Go to kaggle.com → Datasets → New Dataset
2. Upload `lstm_project.zip` as a dataset (name it e.g. `lstm-project-code`)
3. Upload `lstm_raw.zip` as a dataset (name it e.g. `lstm-raw-data`)
4. Create a new Notebook, enable GPU (Settings → Accelerator → GPU)
5. Add both datasets to the notebook (Add Data → search your dataset names)
6. Run all cells top-to-bottom

### What this notebook does
- Copies project code and data from Kaggle input to working directory
- Installs dependencies
- Runs the full pipeline for each dataset (NSL-KDD, UNSW-NB15, CICIDS2017)
- Saves results to `/kaggle/working/` for download

---
## Cell 1 — Check GPU

In [ ]:
import subprocess, os

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("WARNING: No GPU detected. Go to Settings → Accelerator → GPU P100")
    print("Without GPU, training will be very slow.")

---
## Cell 2 — Extract project code and data from Kaggle input

In [ ]:
import shutil

WORKING = "/kaggle/working"
INPUT = "/kaggle/input"

# Find project zip in Kaggle input
project_zip = None
data_zip = None

for root, dirs, files in os.walk(INPUT):
    for f in files:
        path = os.path.join(root, f)
        if f == "lstm_project.zip":
            project_zip = path
        elif f == "lstm_raw.zip":
            data_zip = path

print(f"Project zip: {project_zip}")
print(f"Data zip:    {data_zip}")

# Extract project code
if project_zip:
    print("\nExtracting project code...")
    !unzip -q -o "{project_zip}" -d "{WORKING}"
    print("Done.")
else:
    print("ERROR: lstm_project.zip not found. Add it as a Kaggle dataset.")

# Extract raw data
if data_zip:
    raw_dir = f"{WORKING}/data/raw"
    if not os.path.exists(f"{raw_dir}/nsl_kdd"):
        print("\nExtracting raw data...")
        os.makedirs(raw_dir, exist_ok=True)
        !unzip -q -o "{data_zip}" -d "{raw_dir}"
        print("Done.")
    else:
        print("\nRaw data already extracted.")
else:
    print("ERROR: lstm_raw.zip not found. Add it as a Kaggle dataset.")

print("\nDataset directories:")
!ls {WORKING}/data/raw/

---
## Cell 3 — Rename local tensorflow/ shim + set backend

In [ ]:
os.chdir("/kaggle/working")

# Rename the local tensorflow shim so real TF (GPU) is used
if os.path.exists("tensorflow"):
    !mv tensorflow tensorflow_shim_local
    print("Renamed tensorflow/ shim to tensorflow_shim_local/")
else:
    print("No tensorflow/ shim found (already renamed or not present).")

os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

---
## Cell 4 — Install dependencies

In [ ]:
!pip install -q -r requirements_colab.txt
!pip install -q click
print("Dependencies installed.")

---
## Cell 5 — Verify environment

In [ ]:
import sys, numpy as np, keras

print(f"Python:     {sys.version}")
print(f"NumPy:      {np.__version__}")
print(f"Keras:      {keras.__version__}")
print(f"Backend:    {keras.backend.backend()}")

gpus = tf.config.list_physical_devices("GPU")
print(f"GPU:        {gpus[0].name if gpus else 'NONE'}")

print(f"\nProject files:")
!ls run_pipeline.py requirements_colab.txt 2>&1

---
## Cell 6 — Run Pipeline: NSL-KDD

In [ ]:
import shutil

# Clean previous run artifacts
for d in ["reports", "models"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleaned {d}/")
# Remove stale flat processed files (old bug artifact)
if os.path.exists("data/processed"):
    for item in os.listdir("data/processed"):
        p = f"data/processed/{item}"
        if os.path.isfile(p):
            os.remove(p)
            print(f"Removed stale file: {p}")

os.environ["KERAS_BACKEND"] = "tensorflow"

!python run_pipeline.py --dataset nsl_kdd --skip-eda 2>&1 | tail -60

---
## Cell 7 — Run Pipeline: UNSW-NB15

In [ ]:
# Clean previous run artifacts
for d in ["reports", "models"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleaned {d}/")

os.environ["KERAS_BACKEND"] = "tensorflow"

!python run_pipeline.py --dataset unsw_nb15 --skip-eda 2>&1 | tail -60

---
## Cell 8 — Run Pipeline: CICIDS2017 (largest, ~3-5 hrs on P100)

In [ ]:
# Clean previous run artifacts
for d in ["reports", "models"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleaned {d}/")

os.environ["KERAS_BACKEND"] = "tensorflow"

!python run_pipeline.py --dataset cicids2017 --skip-eda 2>&1 | tail -60

---
## Cell 9 — Package all results for download

In [ ]:
# Copy final results to a single output folder
import shutil, json
from pathlib import Path

OUT = "/kaggle/working/final_results"
os.makedirs(OUT, exist_ok=True)

for ds in ["nsl_kdd", "unsw_nb15", "cicids2017"]:
    for src in ["reports", "models"]:
        if os.path.exists(src):
            dst = f"{OUT}/{ds}/{src}"
            shutil.copytree(src, dst, dirs_exist_ok=True)

# Create download zip
!cd /kaggle/working && zip -r final_results.zip final_results/
print("\nDownload 'final_results.zip' from the Kaggle output panel.")
print("(Right sidebar → Output → final_results.zip → Download)")

---
## Cell 10 — Summary of all runs

In [ ]:
print("=" * 60)
print("RESULTS SUMMARY — All Datasets")
print("=" * 60)

for ds in ["nsl_kdd", "unsw_nb15", "cicids2017"]:
    for subdir in ["reports/metrics", "reports", "metrics"]:
        result_file = Path(f"{OUT}/{ds}/{subdir}/evaluation_results.json")
        if result_file.exists():
            break
    else:
        # Also check working dir
        for subdir in ["reports/metrics", "reports"]:
            result_file = Path(f"/kaggle/working/{subdir}/evaluation_results.json")
            if result_file.exists():
                break
        else:
            result_file = None

    print(f"\n--- {ds.upper()} ---")
    if result_file and result_file.exists():
        with open(result_file) as f:
            data = json.load(f)
        for model_name, metrics in data.items():
            acc = metrics.get("accuracy", "N/A")
            f1 = metrics.get("f1_macro", "N/A")
            roc = metrics.get("roc_auc_macro", "N/A")
            if isinstance(acc, float):
                print(f"  {model_name:25s}  Acc={acc:.4f}  F1={f1:.4f}  ROC={roc:.4f}")
            else:
                print(f"  {model_name:25s}  Acc={acc}  F1={f1}  ROC={roc}")
    else:
        print("  No results found.")